# Deposit Attrition EDA — account level, then customer level

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**
Tracks `PKG_Attrition_EDA_Kickoff_Spec.md` and `PKG_Attrition_Analysis_Brief.md`.

This notebook does **one job**: establish what the deposit and payment data actually
say, so that the signal work in §7–§9 of the analysis brief rests on facts rather
than on the brief's assumptions. It computes no attrition features and fits no model.

**What it answers** — the nine open questions from §6 of the kickoff spec, plus the
two spec claims that are load-bearing for everything downstream (`trans_id` uniquely
identifies a transaction; `cust_pwr_id` is 1:1 with `mdm_id`). Every answer is written
to a findings register that renders as the final cell.

**The correction that reshapes the study.** The brief assumed no closure flag exists
and that the 30%-decline rule *was* the definition of attrition. `acct_status` and
`closed_dt` exist. So §8 stops treating the 30% rule as ground truth and scores it —
alongside six other candidates — against observed closure.

**Conventions**
- Monthly grain. Daily stays available; nothing here forecloses dropping to it.
- Every id is a string end to end. An int64/string mismatch on these joins fails
  silently and produces plausible-looking output.
- `.show()` is never called. Everything renders through `disp()` as pandas.
- Output is batched into **11 numbered blocks**, each dense enough to screenshot whole.

**Before running:** set `TBL_DEPOSITS` in the config cell. It was never specified.

### Output blocks

| Block | Cell | Answers |
|---|---|---|
| 1 | Load + dtype report | source schema, scope, `trans_id` uniqueness |
| 2 | Grain + duplicates | **Q6** duplicate account-days · **Q9** business calendar |
| 3 | Account summary + status codes | **Q5** `acct_status` code space |
| 4 | Closure mechanics | **Q1** life after `closed_dt` · **Q4** CD maturity |
| 5 | `avg_monthly_bal_1` | **Q2** MTD or prior complete month |
| 6 | Monthly panel | account-month panel written to parquet |
| 7 | Proxy scorecard | **Q3** candidate definitions vs the real flag |
| 8 | Payments profile | direction mix, rails, same-name outflow, destination banks |
| 9 | Join coverage | **Q8** payments↔deposits · **Q7** `cust_pwr_id`↔`mdm_id` |
| 10 | Customer roll-up | account closure vs customer attrition |
| 11 | Findings register | every answer, in one table |

## 0 · Configuration

In [ ]:
# =====================================================================
# 0 · CONFIGURATION  — the only cell that should need editing
# =====================================================================
from pathlib import Path

# ── Table locations ───────────────────────────────────────────────────
DB            = "dsihd01p_dsi"
TBL_PAYMENTS  = f"{DB}.neo4j_payments"
TBL_DEPOSITS  = f"{DB}.lap_dsi_universe_optimized"

# ── Scope ─────────────────────────────────────────────────────────────
DATE_START    = "2024-01-01"
DATE_END      = "2026-07-31"

# ── Output: TWO destinations, and they are not interchangeable ────────
# QA tables are small and written with pandas -> must be a LOCAL path.
# Panels are big and written with Spark -> must be an HDFS URI held as a
# plain string. pathlib collapses "hdfs://host/p" into "hdfs:/host/p",
# which Spark then resolves against the HDFS root; the write fails with
# "Permission denied ... inode=/". Never put an HDFS URI inside a Path().
OUT_DIR       = Path("../eda/attrition")                        # LOCAL, csv
HDFS_DIR      = "hdfs://nameservice1/user/pk36814/attrition"    # STRING, parquet

# ── Behaviour ─────────────────────────────────────────────────────────
MAX_ROWS      = 60          # display cap; nothing larger is ever collected
SAMPLE_MOD    = 100         # deterministic 1-in-N account sample for daily-grain work
ZERO_TOL      = 1.0         # |balance| below this counts as zero
CLOSED_CODE   = "C"         # acct_status value meaning closed

# ── Deposit column projection ─────────────────────────────────────────
# The deposit table carries ~70 columns. Only these are analytical. The
# rest (rates, promo codes, load timestamps) are read once for the
# duplicate diagnostic and then dropped, because carrying them in the
# grain is what turns a benign restatement into a "conflict": two rows
# identical on balance and status but differing on curr_int_rate are one
# account-day, not two.
DEP_KEEP = ["acct_full_acct_id", "edw_tda_load_dt", "balance", "avg_monthly_bal_1",
            "acct_status", "acct_status_desc", "deposit_family",
            "opened_dt", "closed_dt", "cust_pwr_id", "cust_name"]
# Included when present. maturity_dt and certificate_number answer Q4
# directly and beat inferring a round term from opened_dt / closed_dt.
DEP_KEEP_OPT = ["maturity_dt", "request_maturity_dt", "certificate_number",
                "sub_product_cd", "sub_product_desc"]

# Deterministic tiebreak for account-days that still conflict after
# projection. First column present wins, descending.
DEDUP_TIEBREAK = ["bdh_hdfs_load_ts", "edw_tda_load_dt"]

# Proxy thresholds under test (§6 Q3). None of these is settled.
P30_DROP      = 0.70        # current rule: avg3 < 0.70 x prior6 avg
P15_DROP      = 0.85        # lower threshold, shorter window
T0_TOP        = 0.95        # brief's "top of the slide": avg3 >= 0.95 x med12
PERSIST_M     = 3           # months a depressed level must hold

# ── Fail-fast thresholds ──────────────────────────────────────────────
MIN_ACCT_JOIN_RATE = 0.50   # payments->deposits account match rate; below this, stop

In [ ]:
# =====================================================================
# 1 · IMPORTS, DISPLAY HELPERS, FINDINGS REGISTER
# =====================================================================
import re
import pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from IPython.display import display, HTML

spark = (SparkSession.builder
         .appName("pkg_attrition_eda")
         .config("spark.sql.shuffle.partitions", "400")
         .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
         .enableHiveSupport()
         .getOrCreate())

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

OUT_DIR.mkdir(parents=True, exist_ok=True)   # local only

def hp(name):
    """HDFS path as a string. Deliberately not pathlib - see config."""
    return f"{HDFS_DIR.rstrip('/')}/{name}"

def preflight_hdfs():
    """Write one row and read it back. A permissions problem discovered
    here costs seconds; discovered at the panel write it costs the run."""
    probe = hp("_preflight")
    try:
        spark.createDataFrame([(1,)], "x int").write.mode("overwrite").parquet(probe)
        spark.read.parquet(probe).count()
        print("hdfs write OK ->", HDFS_DIR)
        return True
    except Exception as e:
        print("!! HDFS WRITE FAILED ->", HDFS_DIR)
        print("  ", str(e).splitlines()[0][:300])
        print("   Fix HDFS_DIR before running section 6. A pathlib-mangled URI "
              "('hdfs:/host/...') resolves to the HDFS root and denies WRITE.")
        return False

# ── Findings register ─────────────────────────────────────────────────
# Every open question from the spec gets an answer written here as it is
# resolved. The last cell renders the whole register as one table, so the
# notebook always ends with an auditable statement of what is now known.
FINDINGS = []

def note(qid, question, answer, detail=""):
    """Record an answer to an open question. Overwrites on re-run."""
    global FINDINGS
    FINDINGS = [f for f in FINDINGS if f["id"] != qid]
    FINDINGS.append(dict(id=qid, question=question, answer=str(answer), detail=str(detail)))

# ── Display ───────────────────────────────────────────────────────────
def disp(obj, title=None, n=None, save=None, transpose=False):
    """Render a Spark or pandas frame AS PANDAS. Never calls .show().

    n     : row cap (defaults to MAX_ROWS) - nothing bigger is collected
    save  : basename; writes OUT_DIR/<save>.csv
    """
    n = MAX_ROWS if n is None else n
    if hasattr(obj, "toPandas"):
        out = obj.limit(n).toPandas()
    else:
        out = obj.copy() if isinstance(obj, pd.DataFrame) else pd.DataFrame(obj)
    if save:
        out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;"
                     f"margin:10px 0 2px;color:#111'>{title}"
                     f"<span style='font-weight:400;color:#888'> &middot; {len(out)} rows</span></div>"))
    display(out.T if transpose else out)
    return out

def kv(d, title=None, save=None):
    """Render a dict of scalars as a two-column table."""
    out = pd.DataFrame({"metric": list(d.keys()), "value": list(d.values())})
    return disp(out, title=title, n=len(out), save=save)

# ── Column hygiene ────────────────────────────────────────────────────
_NULL_TOKENS = ["NULL", "NONE", "NAN", "N/A", "NA", "\\N"]

def nz(c):
    """Trim, cast to string, and normalise empty string / null tokens to NULL.

    Parquet and Hive both let '' and NULL coexist for the same meaning.
    Normalising once here means every downstream test asks only 'is this
    absent' rather than depending on whichever .fillna runs first.
    """
    t = F.trim(F.col(c).cast("string"))
    return F.when(t.isNull() | (t == "") | F.upper(t).isin(_NULL_TOKENS), None).otherwise(t)

def norm_name(c):
    """Case / punctuation / whitespace only. NOT entity resolution -
    no suffix stripping, no fuzzy matching. Per spec, exact match only."""
    x = F.upper(F.trim(F.col(c).cast("string")))
    x = F.regexp_replace(x, "[^A-Z0-9 ]", " ")
    x = F.regexp_replace(x, " +", " ")
    return F.trim(x)

def sample_acct(col="acct_full_acct_id", mod=None):
    """Deterministic 1-in-N account sample. Hash-based, so it is stable
    across runs and across the two tables - a random sample is not."""
    mod = SAMPLE_MOD if mod is None else mod
    return (F.abs(F.hash(F.col(col))) % mod) == 0

def m_idx(datecol):
    """Integer month index. Used as the ORDER BY for rangeBetween windows,
    which handles months with no row correctly - rowsBetween does not."""
    return F.year(datecol) * 12 + F.month(datecol)

def pct(num, den):
    return float(num) / float(den) if den else float("nan")

print("spark", spark.version, "| csv:", OUT_DIR.resolve(), "| parquet:", HDFS_DIR)
HDFS_OK = preflight_hdfs()

## 1 · Load and profile the sources

In [ ]:
# =====================================================================
# 2 · LOAD + SOURCE DTYPE REPORT                       [OUTPUT BLOCK 1]
# =====================================================================
# ID dtype discipline: every id is a string end to end. An int64/string
# mismatch on these joins fails silently and produces plausible output -
# it has happened on this project before.
#
# The deposit table is read twice: `dep_full` keeps all ~70 columns and is
# used only for the duplicate diagnostic in block 2, where "which column
# differs" is the question. `dep` is projected to DEP_KEEP and is what
# everything else runs on.

PAY_ID_COLS = ["trans_id", "mdm_id_pays", "mdm_id_receives",
               "pnc_dep_acct_pays", "pnc_dep_acct_receives", "unq_cpty_acct_id"]
DEP_ID_COLS = ["acct_full_acct_id", "cust_pwr_id", "acct_status", "deposit_family"]

_pay_raw = spark.table(TBL_PAYMENTS)
_dep_raw = spark.table(TBL_DEPOSITS)

def dtype_report(df, name, id_cols):
    rows = []
    have = dict(df.dtypes)
    for c, t in have.items():
        issue = ""
        if c in id_cols and t != "string":
            issue = f"NOT STRING ({t}) - cast in extract; padding/precision may already be lost"
        rows.append(dict(table=name, column=c, source_type=t, issue=issue))
    for c in id_cols:
        if c not in have:
            rows.append(dict(table=name, column=c, source_type="MISSING", issue="COLUMN NOT FOUND"))
    return pd.DataFrame(rows)

schema_rep = pd.concat([dtype_report(_pay_raw, "payments", PAY_ID_COLS),
                        dtype_report(_dep_raw, "deposits", DEP_ID_COLS)], ignore_index=True)

# ── payments ──────────────────────────────────────────────────────────
pay = _pay_raw
for c in PAY_ID_COLS + ["cpty_name", "cpty_fin_entity_name", "customer_name_pays",
                        "customer_name_receives", "payment_rail", "category"]:
    if c in _pay_raw.columns:
        pay = pay.withColumn(c, nz(c))

pay = (pay
       .withColumn("trans_dt", F.to_date(F.col("trans_dt")))
       .filter(F.col("trans_dt").between(F.lit(DATE_START), F.lit(DATE_END)))
       .withColumn("trans_amt", F.col("trans_amt").cast("double"))
       .withColumn("ym", F.date_format("trans_dt", "yyyy-MM"))
       .withColumn("direction",
                   F.when(F.col("mdm_id_pays").isNotNull() & F.col("mdm_id_receives").isNotNull(), "internal_c2c")
                    .when(F.col("mdm_id_pays").isNull()    & F.col("mdm_id_receives").isNotNull(), "inbound_cpty")
                    .when(F.col("mdm_id_pays").isNotNull() & F.col("mdm_id_receives").isNull(),    "outbound_cpty")
                    .otherwise("BOTH_NULL_INVALID"))
       .withColumn("pnc_acct", F.coalesce("pnc_dep_acct_pays", "pnc_dep_acct_receives"))
       .withColumn("pnc_mdm",  F.coalesce("mdm_id_pays", "mdm_id_receives")))
pay.createOrReplaceTempView("pay")

# ── deposits: full, then projected ────────────────────────────────────
def _prep_dep(df):
    for c in DEP_ID_COLS + ["acct_status_desc", "cust_name"]:
        if c in df.columns:
            df = df.withColumn(c, nz(c))
    df = (df.withColumn("edw_tda_load_dt", F.to_date(F.col("edw_tda_load_dt")))
            .filter(F.col("edw_tda_load_dt").between(F.lit(DATE_START), F.lit(DATE_END)))
            .withColumn("balance", F.col("balance").cast("double")))
    for c in ["opened_dt", "closed_dt", "maturity_dt", "request_maturity_dt"]:
        if c in df.columns:
            df = df.withColumn(c, F.to_date(F.col(c)))
    if "avg_monthly_bal_1" in df.columns:
        df = df.withColumn("avg_monthly_bal_1", F.col("avg_monthly_bal_1").cast("double"))
    return (df.withColumn("ym", F.date_format("edw_tda_load_dt", "yyyy-MM"))
              .withColumn("m_idx", m_idx(F.col("edw_tda_load_dt")))
              .withColumn("is_closed_status", (F.col("acct_status") == F.lit(CLOSED_CODE)).cast("int")))

dep_full = _prep_dep(_dep_raw)

DEP_OPT_PRESENT = [c for c in DEP_KEEP_OPT if c in _dep_raw.columns]
DEP_MISSING     = [c for c in DEP_KEEP if c not in _dep_raw.columns]
if DEP_MISSING:
    print("!! DEP_KEEP columns not in the source table:", DEP_MISSING)

_proj = [c for c in DEP_KEEP if c in dep_full.columns] + DEP_OPT_PRESENT + \
        ["ym", "m_idx", "is_closed_status"]
dep = dep_full.select(*_proj)
dep.createOrReplaceTempView("dep")

print(f"deposit columns: {len(_dep_raw.columns)} in source -> {len(dep.columns)} projected"
      f" | optional present: {DEP_OPT_PRESENT or 'none'}")

# ── headline shape ────────────────────────────────────────────────────
p_shape = pay.agg(F.count("*").alias("pay_rows"),
                  F.countDistinct("trans_id").alias("pay_distinct_trans_id"),
                  F.min("trans_dt").alias("pay_min_dt"),
                  F.max("trans_dt").alias("pay_max_dt")).toPandas().iloc[0]
d_shape = dep.agg(F.count("*").alias("dep_rows"),
                  F.countDistinct("acct_full_acct_id").alias("dep_accounts"),
                  F.countDistinct("cust_pwr_id").alias("dep_customers"),
                  F.countDistinct("edw_tda_load_dt").alias("dep_load_dates"),
                  F.min("edw_tda_load_dt").alias("dep_min_dt"),
                  F.max("edw_tda_load_dt").alias("dep_max_dt")).toPandas().iloc[0]

disp(schema_rep[schema_rep.issue != ""], title="1a &middot; Source dtype issues (empty table = clean)",
     n=100, save="qa_source_schema")
kv({**p_shape.to_dict(), **d_shape.to_dict()}, title="1b &middot; Shape in scope", save="qa_shape")

if p_shape.pay_rows != p_shape.pay_distinct_trans_id:
    print(f"!! trans_id NOT UNIQUE: {p_shape.pay_rows - p_shape.pay_distinct_trans_id:,} duplicate rows. "
          f"Spec says it should be. Investigate before using amounts.")
note("SPEC", "trans_id unique / one row per transaction?",
     "YES" if p_shape.pay_rows == p_shape.pay_distinct_trans_id else "NO - duplicates present",
     f"{p_shape.pay_rows:,} rows / {p_shape.pay_distinct_trans_id:,} ids")

## 2 · Deposit grain, duplicates, calendar  — Q6, Q9

In [ ]:
# =====================================================================
# 3 · DEPOSIT GRAIN + DUPLICATE ACCOUNT-DAYS  (Q6)     [OUTPUT BLOCK 2]
# =====================================================================
# Two grains are tested, and the difference between them is the finding.
#
#   FULL      - all ~70 source columns. A pair of rows differing only on
#               curr_int_rate counts as a conflict here.
#   PROJECTED - DEP_KEEP only. This is the grain the panel actually uses,
#               so this is the number that matters.
#
# The first run reported 3.67M conflicting account-days at the full grain.
# If the projected count is far smaller, the conflicts were carried by
# columns we do not analyse and the panel is safe. If it is not, there is
# a real restatement to resolve.

def dup_profile(df, label):
    k = (df.groupBy("acct_full_acct_id", "edw_tda_load_dt")
           .agg(F.count("*").alias("n_rec")).filter("n_rec > 1"))
    n = k.count()
    return k, dict(grain=label,
                   account_days=df.select("acct_full_acct_id", "edw_tda_load_dt").distinct().count(),
                   duplicated=n,
                   max_rec_per_acct_day=(k.agg(F.max("n_rec")).collect()[0][0] if n else 1),
                   accounts_affected=(k.select("acct_full_acct_id").distinct().count() if n else 0))

k_full, g_full = dup_profile(dep_full.dropDuplicates(), "FULL (all source columns)")
k_proj, g_proj = dup_profile(dep.dropDuplicates(),      "PROJECTED (DEP_KEEP)")
disp(pd.DataFrame([g_full, g_proj]),
     title="2a &middot; Grain test at two column sets - the gap is the diagnosis", save="qa_grain")

n_dup_full = g_full["duplicated"]
n_dup_proj = g_proj["duplicated"]

# ── which column actually differs, at the FULL grain ──────────────────
# countDistinct ignores NULL, so coalesce to a sentinel first - otherwise
# a column null on one row and populated on the other reads as identical,
# which is exactly the difference worth seeing.
if n_dup_full:
    dup_rows = dep_full.dropDuplicates().join(k_full.drop("n_rec"),
                                              ["acct_full_acct_id", "edw_tda_load_dt"], "inner").cache()
    cols = [c for c in dep_full.columns if c not in ("acct_full_acct_id", "edw_tda_load_dt")]
    per_group = dup_rows.groupBy("acct_full_acct_id", "edw_tda_load_dt").agg(
        *[F.countDistinct(F.coalesce(F.col(c).cast("string"), F.lit("<NULL>"))).alias(c) for c in cols])
    diffs = per_group.agg(*[F.sum((F.col(c) > 1).cast("int")).alias(c) for c in cols]).toPandas().T
    diffs.columns = ["n_dup_groups_where_column_differs"]
    diffs["share_of_dup_groups"] = diffs.iloc[:, 0] / n_dup_full
    diffs["in_DEP_KEEP"] = [c in DEP_KEEP or c in DEP_KEEP_OPT for c in diffs.index]
    diffs = diffs.sort_values("n_dup_groups_where_column_differs", ascending=False)
    disp(diffs[diffs.iloc[:, 0] > 0].reset_index().rename(columns={"index": "column"}),
         title="2b &middot; Which column differs inside a duplicate account-day "
               "(in_DEP_KEEP=False means it cannot affect the panel)",
         n=100, save="qa_dup_column_diffs")

    ex = (k_full.orderBy(F.col("n_rec").desc()).limit(2)
          .join(dep_full, ["acct_full_acct_id", "edw_tda_load_dt"], "inner")
          .orderBy("acct_full_acct_id", "edw_tda_load_dt"))
    disp(ex, title="2c &middot; Two worst duplicate groups, all source columns", n=12, transpose=True,
         save="qa_dup_examples")
    dup_rows.unpersist()

note("Q6", "Multiple records for the same account on the same load date?",
     f"FULL grain {n_dup_full:,} account-days; PROJECTED grain {n_dup_proj:,}",
     "If PROJECTED is ~0 the conflicts sit in columns we do not analyse (rates, load timestamps) "
     "and the panel is unaffected. Read 2b before accepting the dedup rule in section 6.")

# ── business-date calendar (Q9) ───────────────────────────────────────
cal = (dep.select("edw_tda_load_dt").distinct()
         .withColumn("dow", F.date_format("edw_tda_load_dt", "E"))
         .withColumn("ym", F.date_format("edw_tda_load_dt", "yyyy-MM")))
dow = cal.groupBy("dow").agg(F.count("*").alias("n_dates")).orderBy(F.col("n_dates").desc())
pm = cal.groupBy("ym").agg(F.count("*").alias("n_load_dates")).orderBy("ym").toPandas()

disp(dow, title="2d &middot; Load dates by day of week (weekend rows = not a business calendar)",
     save="qa_calendar_dow")
disp(pm, title="2e &middot; Load dates per month (a short month is a data gap, not a holiday)",
     n=40, save="qa_calendar_month")
note("Q9", "Is edw_tda_load_dt a clean business-date calendar?",
     f"{pm.n_load_dates.min()}-{pm.n_load_dates.max()} load dates/month across {len(pm)} months",
     "Weekend counts in qa_calendar_dow; any month far below ~20 is a gap to explain")

## 3 · Account summary and status code space — Q5

In [ ]:
# =====================================================================
# 4 · ACCOUNT SUMMARY + STATUS CODE SPACE  (Q5)        [OUTPUT BLOCK 3]
# =====================================================================
# One row per account. Everything downstream keys off this.

_base = [
    F.min("edw_tda_load_dt").alias("first_seen"),
    F.max("edw_tda_load_dt").alias("last_seen"),
    F.countDistinct("edw_tda_load_dt").alias("n_days"),
    F.countDistinct("ym").alias("n_months"),
    F.max("opened_dt").alias("opened_dt"),
    F.max("closed_dt").alias("closed_dt"),
    F.countDistinct("closed_dt").alias("n_distinct_closed_dt"),
    F.sum("is_closed_status").alias("n_days_status_C"),
    F.countDistinct("acct_status").alias("n_distinct_status"),
    F.countDistinct("deposit_family").alias("n_distinct_family"),
    F.countDistinct("cust_pwr_id").alias("n_distinct_cust"),
    F.max("cust_pwr_id").alias("cust_pwr_id"),
    F.avg("balance").alias("bal_mean_all"),
    F.min("balance").alias("bal_min"),
    F.max("balance").alias("bal_max"),
    # struct-max = the last record, without a second window pass
    F.max(F.struct("edw_tda_load_dt", "balance", "acct_status",
                   "acct_status_desc", "deposit_family")).alias("_last"),
]
# maturity_dt / certificate_number are carried when the source has them -
# they answer Q4 directly and beat inferring a term from opened/closed.
_extra = [F.max(c).alias(c) for c in DEP_OPT_PRESENT]

acct = (dep.groupBy("acct_full_acct_id").agg(*(_base + _extra))
        .select("*",
                F.col("_last.balance").alias("last_balance"),
                F.col("_last.acct_status").alias("last_status"),
                F.col("_last.acct_status_desc").alias("last_status_desc"),
                F.col("_last.deposit_family").alias("deposit_family"))
        .drop("_last")
        .withColumn("ever_closed_status", (F.col("n_days_status_C") > 0).cast("int"))
        .withColumn("has_closed_dt", F.col("closed_dt").isNotNull().cast("int"))
        .withColumn("days_after_closed_dt", F.datediff("last_seen", "closed_dt"))
        .withColumn("days_before_panel_end", F.datediff(F.lit(DATE_END).cast("date"), F.col("last_seen")))
        ).cache()

n_acct = acct.count()
acct.createOrReplaceTempView("acct")

# ── status code space, with account type in the same table ────────────
status_map = (dep.groupBy("acct_status", "acct_status_desc").agg(
                  F.count("*").alias("n_rows"),
                  F.countDistinct("acct_full_acct_id").alias("n_accts"),
                  F.countDistinct("deposit_family").alias("n_families"),
                  F.avg("balance").alias("mean_balance"),
                  F.avg((F.abs(F.col("balance")) < ZERO_TOL).cast("double")).alias("share_zero_bal"),
                  F.avg(F.col("closed_dt").isNotNull().cast("double")).alias("share_with_closed_dt"),
                  F.min("edw_tda_load_dt").alias("first_seen"),
                  F.max("edw_tda_load_dt").alias("last_seen"))
                .orderBy(F.col("n_rows").desc()))
disp(status_map, title="3a &middot; acct_status code space (all account types)", n=60, save="qa_status_codes")

# ── family x open/closed, compact ─────────────────────────────────────
fam = (acct.groupBy("deposit_family").agg(
           F.count("*").alias("n_accts"),
           F.sum("has_closed_dt").alias("n_with_closed_dt"),
           F.sum("ever_closed_status").alias("n_ever_status_C"),
           F.avg("bal_mean_all").alias("mean_bal"),
           F.avg("n_months").alias("mean_months_observed"))
       .withColumn("share_closed_dt", F.col("n_with_closed_dt") / F.col("n_accts"))
       .orderBy(F.col("n_accts").desc()))
disp(fam, title="3b &middot; deposit_family x closure (quick checkup only - do not go deeper here yet)",
     n=60, save="qa_family_closure")

note("Q5", "acct_status code meanings",
     f"{status_map.count()} distinct (status, desc) pairs",
     "See qa_status_codes.csv; codes other than 'C' with high share_with_closed_dt are the ones to query")
kv({"accounts_in_scope": n_acct,
    "accounts_with_closed_dt": acct.agg(F.sum("has_closed_dt")).collect()[0][0],
    "accounts_ever_status_C": acct.agg(F.sum("ever_closed_status")).collect()[0][0],
    "accounts_with_multiple_status_codes": acct.filter("n_distinct_status > 1").count(),
    "accounts_with_multiple_families": acct.filter("n_distinct_family > 1").count(),
    "accounts_with_multiple_cust_pwr_id": acct.filter("n_distinct_cust > 1").count(),
    "accounts_with_multiple_closed_dt": acct.filter("n_distinct_closed_dt > 1").count()},
   title="3c &middot; Account-level integrity", save="qa_account_integrity")

## 4 · Closure mechanics — Q1, Q4

In [ ]:
# =====================================================================
# 5 · CLOSURE MECHANICS  (Q1 survival after close, Q4 CD maturity)
#                                                     [OUTPUT BLOCK 4]
# =====================================================================
# Q1: if closed_dt is populated, does the account keep appearing daily?
# This decides whether "disappears from the panel" is even usable as a
# closure proxy, and whether closed accounts drag zero balances into
# every monthly average.

after = (dep.join(acct.select("acct_full_acct_id", "last_seen"),
                  "acct_full_acct_id", "inner")   # closed_dt comes from dep - do not join it twice
            .filter(F.col("closed_dt").isNotNull())
            .withColumn("rel_days", F.datediff("edw_tda_load_dt", "closed_dt")))

q1 = {
    "accounts_with_closed_dt":       acct.filter("has_closed_dt = 1").count(),
    "  ...still appearing after":    after.filter("rel_days > 0").select("acct_full_acct_id").distinct().count(),
    "  ...last row ON closed_dt":    acct.filter("has_closed_dt = 1 AND days_after_closed_dt = 0").count(),
    "  ...last row BEFORE closed_dt": acct.filter("has_closed_dt = 1 AND days_after_closed_dt < 0").count(),
    "rows dated after closed_dt":    after.filter("rel_days > 0").count(),
    "  ...share zero balance":       after.filter("rel_days > 0")
                                          .agg(F.avg((F.abs(F.col("balance")) < ZERO_TOL).cast("double")))
                                          .collect()[0][0],
    "  ...share with status C":      after.filter("rel_days > 0")
                                          .agg(F.avg(F.col("is_closed_status").cast("double")))
                                          .collect()[0][0],
}
kv(q1, title="4a &middot; Q1 - does a closed account keep appearing?", save="qa_q1_after_close")

tail = (after.filter("rel_days > 0")
        .withColumn("bucket", F.when(F.col("rel_days") <= 1, "1 day")
                               .when(F.col("rel_days") <= 7, "2-7 d")
                               .when(F.col("rel_days") <= 31, "8-31 d")
                               .when(F.col("rel_days") <= 93, "32-93 d")
                               .otherwise("94+ d"))
        .groupBy("bucket").agg(F.count("*").alias("n_rows"),
                               F.countDistinct("acct_full_acct_id").alias("n_accts"),
                               F.avg("balance").alias("mean_bal"),
                               F.avg((F.abs(F.col("balance")) < ZERO_TOL).cast("double")).alias("share_zero"))
        .orderBy("bucket"))
disp(tail, title="4b &middot; Life after closed_dt, by lag bucket", save="qa_q1_tail")

# ── status reversal: does an account go C and come back? ──────────────
reversal = acct.filter("ever_closed_status = 1 AND last_status <> '%s'" % CLOSED_CODE)
n_rev = reversal.count()
disp(reversal.select("acct_full_acct_id", "deposit_family", "opened_dt", "closed_dt",
                     "first_seen", "last_seen", "n_days", "n_days_status_C",
                     "last_status", "last_status_desc", "last_balance")
             .orderBy(F.col("n_days_status_C").desc()),
     title=f"4c &middot; Accounts that were status C and then were not ({n_rev:,} total)",
     save="qa_status_reversal")

note("Q1", "Does an account with closed_dt keep appearing in the daily panel?",
     f"{q1['  ...still appearing after']:,} of {q1['accounts_with_closed_dt']:,} accounts have rows after closed_dt",
     "If the share is high, 'disappearance' is NOT a closure proxy and closed zero-balance rows must be "
     "excluded from monthly averages or they will manufacture a decline")

# ── Q4: maturity-driven vs customer-driven closure ────────────────────
# A CD closing on maturity is not attrition. Prefer the declared
# maturity_dt where the source carries it; fall back to inferring a round
# term from opened_dt -> closed_dt only where it does not.
HAS_MATURITY = "maturity_dt" in acct.columns

mat = (acct.filter("has_closed_dt = 1")
       .withColumn("months_open", F.months_between("closed_dt", "opened_dt"))
       .withColumn("months_open_r", F.round("months_open").cast("int"))
       .withColumn("is_round_term",
                   (F.abs(F.col("months_open") - F.round("months_open")) < 0.15) &
                   F.round("months_open").isin(3, 6, 9, 12, 18, 24, 30, 36, 48, 60)))

if HAS_MATURITY:
    mat = (mat.withColumn("days_close_to_maturity", F.datediff("closed_dt", "maturity_dt"))
              .withColumn("closed_at_maturity",
                          F.col("maturity_dt").isNotNull() &
                          (F.abs(F.col("days_close_to_maturity")) <= 5))
              .withColumn("closed_before_maturity",
                          F.col("maturity_dt").isNotNull() &
                          (F.col("days_close_to_maturity") < -5)))
else:
    mat = (mat.withColumn("closed_at_maturity", F.col("is_round_term"))
              .withColumn("closed_before_maturity", F.lit(None).cast("boolean")))
    print("!! maturity_dt not in the projection - Q4 falls back to round-term inference")

_agg = [F.count("*").alias("n_closed"),
        F.avg(F.col("closed_at_maturity").cast("double")).alias("share_closed_at_maturity"),
        F.avg(F.col("is_round_term").cast("double")).alias("share_round_term"),
        F.avg("months_open").alias("mean_months_open"),
        F.avg((F.abs(F.col("last_balance")) < ZERO_TOL).cast("double")).alias("share_zero_at_close")]
if HAS_MATURITY:
    _agg.append(F.avg(F.col("closed_before_maturity").cast("double")).alias("share_early_surrender"))

mat_by_fam = mat.groupBy("deposit_family").agg(*_agg).orderBy(F.col("n_closed").desc())
disp(mat_by_fam, title="4d &middot; Q4 - closure shape by account family "
                       "(high share_closed_at_maturity = not attrition)",
     n=60, save="qa_q4_maturity")

if HAS_MATURITY:
    disp(mat.filter("maturity_dt is not null")
            .withColumn("bucket", F.when(F.col("days_close_to_maturity") < -30, "closed >30d early")
                                   .when(F.col("days_close_to_maturity") < -5,  "closed 6-30d early")
                                   .when(F.col("days_close_to_maturity") <= 5,  "AT MATURITY")
                                   .otherwise("closed after maturity"))
            .groupBy("bucket").agg(F.count("*").alias("n_closed"),
                                   F.countDistinct("deposit_family").alias("n_families"),
                                   F.avg("bal_mean_all").alias("mean_bal")),
         title="4e &middot; Closure timing against declared maturity_dt", save="qa_q4_vs_maturity")
else:
    disp(mat.filter("months_open between 0 and 72")
            .groupBy("months_open_r").agg(F.count("*").alias("n_closed")).orderBy("months_open_r"),
         title="4e &middot; Months opened -> closed (spikes at 12/24/36 = maturity)",
         n=73, save="qa_q4_term_hist")

_n_mat = mat.filter(F.col("closed_at_maturity")).count()
note("Q4", "Can maturity-driven closure be separated from voluntary closure?",
     f"{_n_mat:,} of {mat.count():,} closures land at maturity"
     + (" (declared maturity_dt)" if HAS_MATURITY else " (inferred round term - weaker)"),
     "Families with a high share must be excluded from the attrition population, or held as a control. "
     "Early surrender, where it exists, is the interesting case: a customer breaking a CD before term.")

## 5 · What is `avg_monthly_bal_1`? — Q2

In [ ]:
# =====================================================================
# 6 · WHAT IS avg_monthly_bal_1?  (Q2)                 [OUTPUT BLOCK 5]
# =====================================================================
# Two tests, cheapest first.
#
# TEST 1 - variability within a calendar month. A month-to-date figure
#   moves every business day. A prior-complete-month figure is constant
#   for the whole month. This alone usually settles it.
# TEST 2 - fit against three candidate definitions, on a 1-in-N account
#   sample. Whichever has the smallest median absolute % error wins.

samp = dep.filter(sample_acct()).cache()
n_samp = samp.select("acct_full_acct_id").distinct().count()

# ── TEST 1 ────────────────────────────────────────────────────────────
var = (samp.filter(F.col("avg_monthly_bal_1").isNotNull())
       .groupBy("acct_full_acct_id", "ym")
       .agg(F.countDistinct(F.round("avg_monthly_bal_1", 2)).alias("n_distinct_amb"),
            F.countDistinct("edw_tda_load_dt").alias("n_days")))
t1 = (var.agg(
        F.count("*").alias("acct_months"),
        F.avg((F.col("n_distinct_amb") == 1).cast("double")).alias("share_constant_in_month"),
        F.avg((F.col("n_distinct_amb") == F.col("n_days")).cast("double")).alias("share_changes_every_day"),
        F.avg(F.col("n_distinct_amb") / F.col("n_days")).alias("mean_distinct_per_day"))
      .toPandas().iloc[0].to_dict())
kv(t1, title="5a &middot; Q2 test 1 - does avg_monthly_bal_1 move within the month?", save="qa_q2_variability")

# ── TEST 2 ────────────────────────────────────────────────────────────
w_mtd = (Window.partitionBy("acct_full_acct_id", "ym")
         .orderBy("edw_tda_load_dt").rowsBetween(Window.unboundedPreceding, 0))

month_mean = (samp.groupBy("acct_full_acct_id", "ym")
              .agg(F.avg("balance").alias("mean_bal_month"),
                   F.min("m_idx").alias("m_idx")))
prior = (month_mean
         .withColumn("m_idx_join", F.col("m_idx") + 1)
         .select(F.col("acct_full_acct_id"),
                 F.col("m_idx_join").alias("m_idx"),
                 F.col("mean_bal_month").alias("mean_bal_prior_month")))

fit = (samp.select("acct_full_acct_id", "ym", "m_idx", "edw_tda_load_dt", "balance", "avg_monthly_bal_1")
       .withColumn("mtd_mean", F.avg("balance").over(w_mtd))
       .join(month_mean.select("acct_full_acct_id", "ym", "mean_bal_month"), ["acct_full_acct_id", "ym"], "left")
       .join(prior, ["acct_full_acct_id", "m_idx"], "left")
       .filter(F.col("avg_monthly_bal_1").isNotNull() & (F.abs(F.col("avg_monthly_bal_1")) > ZERO_TOL)))

def _err(cand):
    return F.abs(F.col("avg_monthly_bal_1") - F.col(cand)) / F.abs(F.col("avg_monthly_bal_1"))

cands = ["mtd_mean", "mean_bal_month", "mean_bal_prior_month"]
fit2 = fit
for c in cands:
    fit2 = fit2.withColumn(f"err_{c}", _err(c))

t2 = (fit2.agg(*[F.expr(f"percentile_approx(err_{c}, 0.5)").alias(f"median_abs_pct_err__{c}") for c in cands],
               *[F.avg((F.col(f"err_{c}") < 0.01).cast("double")).alias(f"share_within_1pct__{c}") for c in cands],
               F.count("*").alias("n_rows_compared"))
      .toPandas().iloc[0])
t2df = (pd.DataFrame({"candidate": cands,
                      "median_abs_pct_err": [t2[f"median_abs_pct_err__{c}"] for c in cands],
                      "share_within_1pct":  [t2[f"share_within_1pct__{c}"] for c in cands]})
        .sort_values("median_abs_pct_err"))
disp(t2df, title=f"5b &middot; Q2 test 2 - fit against candidates (1-in-{SAMPLE_MOD} accounts, "
                 f"n={n_samp:,}, {int(t2['n_rows_compared']):,} rows)", save="qa_q2_fit")

winner = t2df.iloc[0]
note("Q2", "Is avg_monthly_bal_1 month-to-date or prior complete month?",
     f"Best fit: {winner.candidate} (median abs err {winner.median_abs_pct_err:.3%})",
     f"Constant within month on {t1['share_constant_in_month']:.1%} of account-months; "
     f"changes daily on {t1['share_changes_every_day']:.1%}. "
     "Constant + best fit to prior month => prior complete month. Daily + best fit to mtd_mean => month-to-date.")

samp.unpersist()

## 6 · Monthly account panel

In [ ]:
# =====================================================================
# 7 · MONTHLY ACCOUNT PANEL                            [OUTPUT BLOCK 6]
# =====================================================================
# Monthly grain per the spec. Daily is available and stays available -
# nothing here forecloses dropping to it later.
#
# Duplicate account-days: exact duplicates are dropped for free by
# distinct(). Only genuine conflicts at the PROJECTED grain reach the
# tiebreak, which keeps the latest-loaded row - a restatement should win
# over the record it restates. Read block 2b before accepting this.

_key = ["acct_full_acct_id", "edw_tda_load_dt"]
_others = [c for c in dep.columns if c not in _key]

dep1 = dep.dropDuplicates()          # exact duplicates gone, no choice made
_still_dup = dep1.groupBy(*_key).count().filter("count > 1").count()

if _still_dup:
    _ties = [c for c in DEDUP_TIEBREAK if c in dep_full.columns]
    if _ties:
        # Deterministic: keep the latest-loaded row. A restatement should win
        # over the record it restates, which max-struct does not guarantee.
        print(f"!! {_still_dup:,} account-days conflict at the projected grain; "
              f"keeping latest by {_ties[0]}")
        # Rank on dep_full so the tiebreak value belongs to the row it
        # selects. Joining the tiebreak back onto dep1 would fan out and
        # then pick a value from a different source row.
        _cols = list(dep.columns)
        _src = (dep_full.select(*_cols, *[c for c in _ties if c not in _cols])
                        .dropDuplicates())
        _w = Window.partitionBy(*_key).orderBy(*[F.col(c).desc_nulls_last() for c in _ties])
        dep1 = (_src.withColumn("_rn", F.row_number().over(_w))
                    .filter("_rn = 1").select(*_cols))
    else:
        print(f"!! {_still_dup:,} account-days conflict and no tiebreak column is present; "
              f"applying arbitrary max-struct. Read block 2b before trusting the panel.")
        dep1 = (dep1.groupBy(*_key)
                .agg(F.max(F.struct(*_others)).alias("_r"))
                .select(*_key, *[F.col(f"_r.{c}").alias(c) for c in _others]))
else:
    print("account-day grain is clean at the projected column set")

# ── build ─────────────────────────────────────────────────────────────
open_row = (F.col("is_closed_status") == 0) & \
           (F.col("closed_dt").isNull() | (F.col("edw_tda_load_dt") <= F.col("closed_dt")))

panel = (dep1.groupBy("acct_full_acct_id", "ym").agg(
             F.min("m_idx").alias("m_idx"),
             F.count("*").alias("n_days"),
             F.avg("balance").alias("avg_bal"),
             F.avg(F.when(open_row, F.col("balance"))).alias("avg_bal_open"),
             F.sum(F.when(open_row, F.lit(1)).otherwise(0)).alias("n_days_open"),
             F.min("balance").alias("min_bal"),
             F.max("balance").alias("max_bal"),
             F.max("is_closed_status").alias("any_status_C"),
             F.max(F.when(F.col("closed_dt").isNotNull() &
                          (F.date_format("closed_dt", "yyyy-MM") == F.col("ym")), 1).otherwise(0)
                   ).alias("closed_this_month"),
             F.max("closed_dt").alias("closed_dt"),
             F.max("opened_dt").alias("opened_dt"),
             F.max("cust_pwr_id").alias("cust_pwr_id"),
             F.max("deposit_family").alias("deposit_family"),
             F.max(F.struct("edw_tda_load_dt", "balance", "acct_status",
                            "avg_monthly_bal_1")).alias("_eom"))
         .select("*",
                 F.col("_eom.edw_tda_load_dt").alias("eom_date"),
                 F.col("_eom.balance").alias("eom_bal"),
                 F.col("_eom.acct_status").alias("eom_status"),
                 F.col("_eom.avg_monthly_bal_1").alias("amb_last"))
         .drop("_eom"))

panel.write.mode("overwrite").parquet(hp("panel_account_month.parquet"))
panel = spark.read.parquet(hp("panel_account_month.parquet")).cache()
panel.createOrReplaceTempView("panel")

cov = (panel.groupBy("ym").agg(
           F.countDistinct("acct_full_acct_id").alias("n_accts"),
           F.avg("n_days").alias("mean_days_observed"),
           F.sum("closed_this_month").alias("n_closed_this_month"),
           F.sum("any_status_C").alias("n_with_C_day"),
           F.avg("avg_bal").alias("mean_avg_bal"),
           F.expr("percentile_approx(avg_bal, 0.5)").alias("median_avg_bal"),
           F.avg((F.abs(F.col("avg_bal")) < ZERO_TOL).cast("double")).alias("share_zero_bal"),
           F.avg((F.col("avg_bal") < 0).cast("double")).alias("share_negative_bal"))
       .orderBy("ym"))
disp(cov, title="6a &middot; Monthly panel coverage (a step change in n_accts is a data event, not attrition)",
     n=40, save="panel_coverage_by_month")

kv({"panel_rows": panel.count(),
    "accounts": panel.select("acct_full_acct_id").distinct().count(),
    "customers": panel.select("cust_pwr_id").distinct().count(),
    "months": panel.select("ym").distinct().count(),
    "acct_months_with_avg_bal_null": panel.filter("avg_bal is null").count(),
    "acct_months_where_open_subset_differs":
        panel.filter("n_days_open < n_days").count()},
   title="6b &middot; Panel shape", save="panel_shape")

## 7 · Candidate attrition definitions vs the closure flag — Q3

In [ ]:
# =====================================================================
# 8 · CLOSURE DEFINITIONS vs THE REAL FLAG  (Q3)       [OUTPUT BLOCK 7]
# =====================================================================
# The brief was written assuming no closure flag exists. One does. So the
# 30% rule stops being the definition of attrition and becomes one
# candidate among several, scored against observed closure.
#
# Read this as a lead/lag study, not a classifier bake-off: a decline rule
# and a closure flag measure different things. What matters is how much
# warning each candidate gives before the flag fires, and at what cost in
# accounts that never close.

BAL = "avg_bal_open"     # switch to "avg_bal" to include post-closure zero rows

w_ord   = Window.partitionBy("acct_full_acct_id").orderBy("m_idx")
w3      = w_ord.rangeBetween(-2, 0)
w6prior = w_ord.rangeBetween(-8, -3)
w6      = w_ord.rangeBetween(-5, 0)
w12     = w_ord.rangeBetween(-11, 0)

p = (panel.withColumn("bal", F.col(BAL))
     .withColumn("avg3",       F.avg("bal").over(w3))
     .withColumn("n3",         F.count("bal").over(w3))
     .withColumn("prior6",     F.avg("bal").over(w6prior))
     .withColumn("n_prior6",   F.count("bal").over(w6prior))
     .withColumn("n_hist",     F.count("bal").over(w12))
     .withColumn("_arr12",     F.array_sort(F.collect_list("bal").over(w12))))

# trailing-12 median without percentile_approx (not a window function here)
p = (p.withColumn("med12", F.expr("element_at(_arr12, cast(size(_arr12)/2 as int) + 1)"))
       .drop("_arr12"))

# month-over-month decline, contiguity-checked so a gap is not a decline
p = (p.withColumn("_lag_bal", F.lag("bal").over(w_ord))
       .withColumn("_lag_m",  F.lag("m_idx").over(w_ord))
       .withColumn("is_dec", F.when((F.col("m_idx") - F.col("_lag_m") == 1) &
                                    (F.col("bal") < F.col("_lag_bal")), 1).otherwise(0))
       .withColumn("n_dec3", F.sum("is_dec").over(w3))
       .withColumn("n_obs3", F.count("*").over(w3))
       # windowed approximation to CUSUM: cumulative relative drift below the
       # trailing-12 median over six months. True recursive CUSUM needs a
       # grouped-map UDF and is deferred.
       .withColumn("drift6", F.sum(F.when(F.col("med12") > ZERO_TOL,
                                          (F.col("bal") - F.col("med12")) / F.col("med12"))
                                    .otherwise(F.lit(0.0))).over(w6)))

_last_m = panel.agg(F.max("m_idx")).collect()[0][0]
p = p.withColumn("_acct_last_m", F.max("m_idx").over(Window.partitionBy("acct_full_acct_id")))

usable = (F.col("n_hist") >= 12)

PROX = {
 "p_30rule":   usable & (F.col("n_prior6") >= 4) & (F.col("n3") >= 3) &
               (F.col("avg3") < P30_DROP * F.col("prior6")),
 "p_15_3m":    usable & (F.col("n3") >= 3) & (F.col("avg3") < P15_DROP * F.col("med12")),
 "p_t0_slide": usable & (F.col("n3") >= 3) & (F.col("avg3") < T0_TOP  * F.col("med12")),
 "p_mom3":     usable & (F.col("n_dec3") == 3) & (F.col("n_obs3") == 3),
 "p_drift6":   usable & (F.col("drift6") < -1.5),
 "p_zero3":    usable & (F.col("n3") >= 3) & (F.abs(F.col("avg3")) < ZERO_TOL),
 "p_gone":     (F.col("m_idx") == F.col("_acct_last_m")) & (F.col("m_idx") < F.lit(_last_m)),
}
for k, cond in PROX.items():
    p = p.withColumn(k, F.when(cond, 1).otherwise(0))
p = p.cache()

# ── truth + first fire per account ────────────────────────────────────
truth = (panel.groupBy("acct_full_acct_id").agg(
             F.min(F.when(F.col("closed_this_month") == 1, F.col("m_idx"))).alias("close_m_dt"),
             F.min(F.when(F.col("any_status_C") == 1, F.col("m_idx"))).alias("close_m_status"),
             F.max("m_idx").alias("acct_last_m"),
             F.count("*").alias("n_months"))
         .withColumn("close_m", F.least(F.coalesce("close_m_dt", F.lit(9999)),
                                        F.coalesce("close_m_status", F.lit(9999))))
         .withColumn("close_m", F.when(F.col("close_m") == 9999, None).otherwise(F.col("close_m")))
         .withColumn("is_closed", F.col("close_m").isNotNull().cast("int")))

fire = (p.groupBy("acct_full_acct_id")
        .agg(*[F.min(F.when(F.col(k) == 1, F.col("m_idx"))).alias(f"f_{k}") for k in PROX]))

ev = truth.join(fire, "acct_full_acct_id", "left").filter("n_months >= 12").cache()
n_ev = ev.count()
n_closed = ev.agg(F.sum("is_closed")).collect()[0][0]

# ── one agg, all proxies ──────────────────────────────────────────────
aggs = []
for k in PROX:
    f = F.col(f"f_{k}")
    fired = f.isNotNull()
    tp    = fired & F.col("is_closed").cast("boolean") & (F.col("close_m") >= f) & (F.col("close_m") - f <= 12)
    aggs += [
        F.sum(fired.cast("int")).alias(f"n_fired__{k}"),
        F.sum(tp.cast("int")).alias(f"n_tp__{k}"),
        F.sum((fired & (F.col("is_closed") == 0)).cast("int")).alias(f"n_fp_never_closed__{k}"),
        F.sum(((F.col("is_closed") == 1) & fired & (f <= F.col("close_m"))).cast("int")).alias(f"n_caught__{k}"),
        F.expr(f"percentile_approx(case when f_{k} is not null and close_m is not null "
               f"and close_m >= f_{k} then close_m - f_{k} end, 0.5)").alias(f"med_lead__{k}"),
    ]
raw = ev.agg(*aggs).toPandas().iloc[0]

rows = []
for k in PROX:
    nf, tp = raw[f"n_fired__{k}"], raw[f"n_tp__{k}"]
    rows.append(dict(
        proxy=k,
        n_fired=int(nf),
        share_of_accounts=pct(nf, n_ev),
        precision_closes_within_12m=pct(tp, nf),
        recall_of_closed=pct(raw[f"n_caught__{k}"], n_closed),
        n_fired_never_closed=int(raw[f"n_fp_never_closed__{k}"]),
        median_lead_months=raw[f"med_lead__{k}"]))
prox_tbl = pd.DataFrame(rows).sort_values("precision_closes_within_12m", ascending=False)

disp(prox_tbl, title=f"7a &middot; Q3 - candidate attrition definitions vs observed closure "
                     f"(n={n_ev:,} accounts with 12+ months, {n_closed:,} closed)",
     n=20, save="q3_proxy_scorecard")

# ── pairwise overlap with the truth flag, and with each other ─────────
ov = ev.agg(*[F.sum(((F.col(f"f_{a}").isNotNull()) & (F.col(f"f_{b}").isNotNull())).cast("int")).alias(f"{a}|{b}")
              for a in PROX for b in PROX if a <= b]).toPandas().iloc[0]
om = pd.DataFrame(0, index=list(PROX), columns=list(PROX))
for kk, v in ov.items():
    a, b = kk.split("|")
    om.loc[a, b] = om.loc[b, a] = int(v)
disp(om.reset_index().rename(columns={"index": "proxy"}),
     title="7b &middot; Accounts fired by both proxies (diagonal = fired at all)", n=20, save="q3_proxy_overlap")

note("Q3", "Relationship between the closure flag, closed_dt and behavioural proxies",
     f"Best precision: {prox_tbl.iloc[0].proxy} at {prox_tbl.iloc[0].precision_closes_within_12m:.1%}",
     "See q3_proxy_scorecard.csv. Lead is capped by the 12-month burn-in: the panel starts "
     f"{DATE_START}, so the first eligible fire month is 2024-12. Pulling deposits back to 2023-01 "
     "would recover a year of evaluable months.")
note("BURN", "Is the 2024-01 start enough for a 12-month baseline?",
     "NO - burn-in consumes the whole of 2024",
     "The brief asked for Jan 2023 onward for exactly this reason. Re-pull if the lead-time numbers matter.")

## 8 · Payments profile

In [ ]:
# =====================================================================
# 9 · PAYMENTS PROFILE                                 [OUTPUT BLOCK 8]
# =====================================================================
pay = pay.cache()

dir_mix = (pay.groupBy("direction").agg(
               F.count("*").alias("n_trans"),
               F.sum("trans_amt").alias("amount"),
               F.countDistinct("pnc_mdm").alias("n_pnc_mdm"),
               F.countDistinct("unq_cpty_acct_id").alias("n_cpty_accts"),
               F.avg(F.col("unq_cpty_acct_id").isNotNull().cast("double")).alias("share_with_cpty_id"),
               F.avg(F.col("cpty_name").isNotNull().cast("double")).alias("share_with_cpty_name"),
               F.avg(F.col("cpty_fin_entity_name").isNotNull().cast("double")).alias("share_with_fin_entity"))
           .withColumn("share_trans", F.col("n_trans") / F.sum("n_trans").over(Window.partitionBy()))
           .orderBy(F.col("n_trans").desc()))
disp(dir_mix, title="8a &middot; Direction mix + counterparty field coverage", save="pay_direction_mix")

rail = (pay.groupBy("payment_rail", "category", "direction").agg(
            F.count("*").alias("n_trans"),
            F.sum("trans_amt").alias("amount"),
            F.expr("percentile_approx(trans_amt, 0.5)").alias("median_amt"))
        .orderBy(F.col("n_trans").desc()))
disp(rail, title="8b &middot; payment_rail x category x direction (top 40 by volume)", n=40, save="pay_rail_mix")

# ── monthly volume, to spot ingestion gaps before reading any trend ───
mo = (pay.groupBy("ym").agg(
          F.count("*").alias("n_trans"),
          F.sum("trans_amt").alias("amount"),
          F.countDistinct("pnc_mdm").alias("n_active_mdm"),
          F.sum(F.when(F.col("direction") == "outbound_cpty", 1).otherwise(0)).alias("n_outbound"),
          F.sum(F.when(F.col("direction") == "inbound_cpty", 1).otherwise(0)).alias("n_inbound"),
          F.sum(F.when(F.col("direction") == "internal_c2c", 1).otherwise(0)).alias("n_internal"))
      .orderBy("ym"))
disp(mo, title="8c &middot; Payments by month (a dip here is ingestion, not behaviour)", n=40, save="pay_by_month")

# ── same-name outflow: exact match only, per spec ─────────────────────
# No suffix stripping, no fuzzy matching. Two variants reported so the
# cost of the light normalisation is visible rather than assumed.
out = (pay.filter("direction = 'outbound_cpty'")
       .withColumn("m_raw", (F.col("cpty_name") == F.col("customer_name_pays")).cast("int"))
       .withColumn("m_norm", (norm_name("cpty_name") == norm_name("customer_name_pays")).cast("int"))
       .withColumn("has_both", (F.col("cpty_name").isNotNull() &
                                F.col("customer_name_pays").isNotNull()).cast("int")))
nm = out.agg(
        F.count("*").alias("n_outbound"),
        F.sum("has_both").alias("n_comparable"),
        F.sum("m_raw").alias("n_match_raw"),
        F.sum("m_norm").alias("n_match_norm"),
        F.sum(F.when(F.col("m_norm") == 1, F.col("trans_amt"))).alias("amt_match_norm"),
        F.sum("trans_amt").alias("amt_outbound"),
        F.countDistinct(F.when(F.col("m_norm") == 1, F.col("mdm_id_pays"))).alias("n_customers_with_same_name_outflow")
      ).toPandas().iloc[0].to_dict()
nm["match_rate_norm_by_count"] = pct(nm["n_match_norm"], nm["n_comparable"])
nm["match_rate_norm_by_amount"] = pct(nm["amt_match_norm"], nm["amt_outbound"])
kv(nm, title="8d &middot; same_name_outflow - exact match, outbound only", save="pay_same_name_match")

# ── where does outbound money go ──────────────────────────────────────
fin = (pay.filter("direction = 'outbound_cpty' AND cpty_fin_entity_name is not null")
       .groupBy("cpty_fin_entity_name").agg(
           F.sum("trans_amt").alias("amount"),
           F.count("*").alias("n_trans"),
           F.countDistinct("mdm_id_pays").alias("n_customers"))
       .orderBy(F.col("amount").desc()))
disp(fin, title="8e &middot; Outbound destination institutions (this IS the fi_destination list - "
                "no external name list needed)", n=40, save="pay_fin_entities")

note("CPTY", "Counterparty name coverage on outbound transactions",
     f"{nm.get('n_comparable', 0) / max(nm.get('n_outbound', 1), 1):.1%} of outbound rows have both names",
     f"Exact-match same-name outflow: {nm['match_rate_norm_by_count']:.3%} of comparable rows, "
     f"{nm['match_rate_norm_by_amount']:.3%} of outbound dollars")
note("FI", "Financial institution name list for fi_destination_flag",
     "NOT NEEDED - cpty_fin_entity_name is populated in the source",
     "The brief listed this as an open item; the staging table already carries the destination bank")

## 9 · Join coverage and id relationships — Q7, Q8

In [ ]:
# =====================================================================
# 10 · JOIN COVERAGE + ID RELATIONSHIPS  (Q7, Q8)      [OUTPUT BLOCK 9]
# =====================================================================
# Two claims from the spec are tested here:
#   - every mdm_id in the payment network has a deposit account
#   - cust_pwr_id is one-to-one with mdm_id
# Both are load-bearing for the whole study. Neither is assumed.

dep_accts = dep.select(F.col("acct_full_acct_id").alias("acct")).distinct()

pay_accts = (pay.select(F.col("pnc_dep_acct_pays").alias("acct")).filter("acct is not null")
             .union(pay.select(F.col("pnc_dep_acct_receives").alias("acct")).filter("acct is not null"))
             .distinct())

# account-id format check before blaming the join
fmt = (pay_accts.withColumn("src", F.lit("payments")).union(
       dep_accts.withColumn("src", F.lit("deposits")))
       .groupBy("src").agg(F.countDistinct("acct").alias("n_accts"),
                           F.min(F.length("acct")).alias("min_len"),
                           F.max(F.length("acct")).alias("max_len"),
                           F.expr("percentile_approx(length(acct), 0.5)").alias("median_len"),
                           F.avg(F.col("acct").rlike("^[0-9]+$").cast("double")).alias("share_all_digits"),
                           F.avg(F.col("acct").startswith("0").cast("double")).alias("share_leading_zero")))
disp(fmt, title="9a &middot; Account id format on both sides (a length mismatch is a padding bug, not a coverage gap)",
     save="join_acct_format")

j = pay_accts.join(dep_accts.withColumn("in_dep", F.lit(1)), "acct", "left")
n_pay_acct = pay_accts.count()
n_matched = j.filter("in_dep = 1").count()
match_rate = pct(n_matched, n_pay_acct)

# fail-fast
if match_rate < MIN_ACCT_JOIN_RATE:
    print(f"!! ACCOUNT JOIN RATE {match_rate:.1%} < {MIN_ACCT_JOIN_RATE:.0%}. "
          f"Check 9a for padding/dtype before interpreting anything below.")

# ── mdm_id -> deposit account, through the payment table ──────────────
mdm_acct = (pay.select(F.col("mdm_id_pays").alias("mdm"), F.col("pnc_dep_acct_pays").alias("acct"))
            .union(pay.select(F.col("mdm_id_receives").alias("mdm"), F.col("pnc_dep_acct_receives").alias("acct")))
            .filter("mdm is not null").distinct())

mdm_cov = (mdm_acct.join(dep.select(F.col("acct_full_acct_id").alias("acct"),
                                    "cust_pwr_id").distinct(), "acct", "left")
           .groupBy("mdm").agg(F.countDistinct("acct").alias("n_accts"),
                               F.countDistinct("cust_pwr_id").alias("n_cust_pwr_id"),
                               F.sum(F.col("cust_pwr_id").isNotNull().cast("int")).alias("n_matched_accts"))).cache()

n_mdm = mdm_cov.count()
kv({"payment_pnc_accounts": n_pay_acct,
    "  ...found in deposits": n_matched,
    "  ...match rate": match_rate,
    "deposit_accounts": dep_accts.count(),
    "  ...never seen in payments": dep_accts.join(pay_accts.withColumn("in_pay", F.lit(1)),
                                                  "acct", "left").filter("in_pay is null").count(),
    "payment mdm_ids": n_mdm,
    "  ...with >=1 matched deposit account": mdm_cov.filter("n_matched_accts > 0").count(),
    "  ...with NO deposit account": mdm_cov.filter("n_matched_accts = 0").count(),
    "  ...mapping to >1 cust_pwr_id": mdm_cov.filter("n_cust_pwr_id > 1").count()},
   title="9b &middot; Q8 - payments <-> deposits coverage", save="join_coverage")

# ── Q7: cust_pwr_id <-> mdm_id, both directions ───────────────────────
link = (mdm_acct.join(dep.select(F.col("acct_full_acct_id").alias("acct"), "cust_pwr_id").distinct(),
                      "acct", "inner")
        .select("mdm", "cust_pwr_id").distinct()).cache()

per_mdm  = link.groupBy("mdm").agg(F.countDistinct("cust_pwr_id").alias("n"))
per_cust = link.groupBy("cust_pwr_id").agg(F.countDistinct("mdm").alias("n"))

q7 = {
    "linked pairs":                       link.count(),
    "distinct mdm_id":                       per_mdm.count(),
    "  ...mdm -> exactly 1 cust_pwr_id":      per_mdm.filter("n = 1").count(),
    "  ...mdm -> more than 1 cust_pwr_id":    per_mdm.filter("n > 1").count(),
    "  ...max cust_pwr_id per mdm_id":        per_mdm.agg(F.max("n")).collect()[0][0],
    "distinct cust_pwr_id":                   per_cust.count(),
    "  ...cust_pwr_id -> exactly 1 mdm":      per_cust.filter("n = 1").count(),
    "  ...cust_pwr_id -> more than 1 mdm":    per_cust.filter("n > 1").count(),
    "  ...max mdm_id per cust_pwr_id":        per_cust.agg(F.max("n")).collect()[0][0],
}
kv(q7, title="9c &middot; Q7 - is cust_pwr_id one-to-one with mdm_id?", save="join_id_cardinality")

disp(link.join(per_mdm.filter("n > 1").select("mdm"), "mdm", "inner").orderBy("mdm"),
     title="9d &middot; Violations: one mdm_id, several cust_pwr_id", n=30, save="join_id_violations")

note("Q8", "Does every mdm_id in the payment network have a deposit account?",
     f"{mdm_cov.filter('n_matched_accts = 0').count():,} of {n_mdm:,} have none "
     f"(account join rate {match_rate:.1%})", "See 9a first - a padding mismatch looks identical to a coverage gap")
note("Q7", "Is cust_pwr_id one-to-one with mdm_id?",
     "YES - one-to-one in both directions"
     if (q7["  ...mdm -> more than 1 cust_pwr_id"] == 0 and
         q7["  ...cust_pwr_id -> more than 1 mdm"] == 0)
     else f"NO - {q7['  ...mdm -> more than 1 cust_pwr_id']:,} mdm_id fan out, "
          f"{q7['  ...cust_pwr_id -> more than 1 mdm']:,} cust_pwr_id fan out",
     "Resolve before any customer-level roll-up; a fanout silently double-counts balances")

# ── study population: customers with BOTH deposit and payment data ────
study = (link.select("cust_pwr_id").distinct()
         .join(panel.select("cust_pwr_id").distinct(), "cust_pwr_id", "inner"))
kv({"deposit customers": panel.select("cust_pwr_id").distinct().count(),
    "payment-visible customers": link.select("cust_pwr_id").distinct().count(),
    "STUDY POPULATION (both)": study.count()},
   title="9e &middot; Study population - counterparties are scoped to these customers only",
   save="study_population")

## 10 · Customer-level roll-up, and the findings register

In [ ]:
# =====================================================================
# 11 · CUSTOMER-LEVEL ROLL-UP                         [OUTPUT BLOCK 10]
# =====================================================================
# Account level first, then this. The roll-up rule is the thing to argue
# about: a customer with one closed account of five has not attrited, and
# summing balances across a customer whose accounts open and close hides
# exactly the event we are trying to detect.

cust = (panel.groupBy("cust_pwr_id", "ym").agg(
            F.min("m_idx").alias("m_idx"),
            F.countDistinct("acct_full_acct_id").alias("n_accts"),
            F.sum(F.when(F.col("any_status_C") == 0, 1).otherwise(0)).alias("n_accts_open"),
            F.sum("avg_bal").alias("total_avg_bal"),
            F.sum("avg_bal_open").alias("total_avg_bal_open"),
            F.sum("eom_bal").alias("total_eom_bal"),
            F.sum("closed_this_month").alias("n_closed_this_month"),
            F.countDistinct("deposit_family").alias("n_families"))
        .withColumn("all_accts_closed", (F.col("n_accts_open") == 0).cast("int")))

cust.write.mode("overwrite").parquet(hp("panel_customer_month.parquet"))
cust = spark.read.parquet(hp("panel_customer_month.parquet")).cache()

c_cov = (cust.groupBy("ym").agg(
             F.countDistinct("cust_pwr_id").alias("n_customers"),
             F.avg("n_accts").alias("mean_accts_per_cust"),
             F.expr("percentile_approx(total_avg_bal, 0.5)").alias("median_total_bal"),
             F.sum("n_closed_this_month").alias("accts_closed"),
             F.sum("all_accts_closed").alias("customers_fully_closed"))
         .orderBy("ym"))
disp(c_cov, title="10a &middot; Customer-month panel coverage", n=40, save="panel_customer_coverage")

# how much of account-level closure is actually customer-level attrition
mix = (panel.filter("closed_this_month = 1")
       .join(cust.select("cust_pwr_id", "ym", "n_accts", "n_accts_open"), ["cust_pwr_id", "ym"], "left")
       .withColumn("cls", F.when(F.col("n_accts_open") == 0, "customer left entirely")
                           .when(F.col("n_accts") > 1, "one of several accounts closed")
                           .otherwise("sole account closed, others open?"))
       .groupBy("cls").agg(F.count("*").alias("n_account_closures"),
                           F.countDistinct("cust_pwr_id").alias("n_customers")))
disp(mix, title="10b &middot; Account closure is not customer attrition - the split",
     save="closure_account_vs_customer")

kv({"customer-months": cust.count(),
    "customers": cust.select("cust_pwr_id").distinct().count(),
    "customers ever fully closed": cust.filter("all_accts_closed = 1").select("cust_pwr_id").distinct().count(),
    "customers with >1 account": cust.filter("n_accts > 1").select("cust_pwr_id").distinct().count(),
    "customers with >1 deposit_family": cust.filter("n_families > 1").select("cust_pwr_id").distinct().count()},
   title="10c &middot; Customer-level shape", save="panel_customer_shape")

note("ROLLUP", "Customer-level roll-up rule",
     "Sum of account avg_bal, with an open-accounts-only variant carried alongside",
     "10b shows how much account closure is customer attrition vs internal account churn. "
     "If 'one of several accounts closed' dominates, the account-level label is the wrong unit for the study.")

# =====================================================================
# 12 · FINDINGS REGISTER                              [OUTPUT BLOCK 11]
# =====================================================================
reg = pd.DataFrame(FINDINGS)
order = ["SPEC", "Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8", "Q9", "BURN", "CPTY", "FI", "ROLLUP"]
reg["_o"] = reg["id"].apply(lambda x: order.index(x) if x in order else 99)
reg = reg.sort_values("_o").drop(columns="_o")
disp(reg, title="11 &middot; OPEN QUESTIONS - answers from this run", n=40, save="FINDINGS")

print("\nWritten to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*.csv")):
    print("  ", f.name)

---

## Next, once this run is read

1. **Fix the dedup rule.** Block 2b names the column that differs inside a duplicate
   account-day. Until that is read, Block 6 applies an arbitrary max-struct tiebreak
   and says so.
2. **Settle the attrition label** from Block 7 and Block 10b. Account-level closure and
   customer-level attrition are different events; 10b shows how different.
3. **Re-pull deposits from 2023-01.** The 12-month burn-in consumes the whole of 2024,
   which leaves roughly twenty evaluable months. The brief asked for 2023 for this reason.
4. **Then** build the Group A–F features from §7 of the analysis brief, with the
   observation frame and blackout rule from §6.4 — nothing at or after `t+1` in the
   features, ever.

Two open items from the brief are already closed by the source data: `cpty_fin_entity_name`
removes the need for an external financial-institution name list, and `acct_status`
removes the need to treat balance decline as a stand-in for departure.